# Parquet Basics — 02: Writing Parquet Files

## What you will learn
Writing Parquet correctly has a major impact on query performance.
Small decisions at write time (compression, row group size, sort order)
affect every future read.

In this notebook:
1. Basic write — modes and options
2. Compression codecs — snappy vs zstd vs gzip vs lz4
3. Row group size tuning — the trade-off between skipping and throughput
4. Controlling number of output files — coalesce vs repartition
5. Writing sorted data — maximising data skipping
6. Schema control — enforcing types and nullability at write


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:05:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:05:42 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:05:44 WARN TaskSetManager: Stage 1 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — Basic Write and Save Modes


In [2]:
OUT = f"{DATA_DIR}/write_test"

# Default write — snappy compression, one file per partition
df.write.mode("overwrite").parquet(f"{OUT}/default")

import glob as glib
files = glib.glob(f"{OUT}/default/*.parquet")
total_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
print(f"Default write:")
print(f"  Files : {len(files)}")
print(f"  Size  : {total_mb:.1f} MB")
print(f"  Rows  : {spark.read.parquet(f'{OUT}/default').count():,}")
print()

# Save modes
for mode in ["overwrite", "append", "ignore"]:
    try:
        df.limit(100).write.mode(mode).parquet(f"{OUT}/mode_test")
        count = spark.read.parquet(f"{OUT}/mode_test").count()
        print(f"  mode={mode:<10} → {count:,} rows")
    except Exception as e:
        print(f"  mode={mode:<10} → {type(e).__name__}")

# error mode (default) — fails if path exists
try:
    df.limit(100).write.parquet(f"{OUT}/mode_test")
except Exception as e:
    print(f"  mode=error      → {type(e).__name__} (path exists)")

26/04/10 21:05:45 WARN TaskSetManager: Stage 4 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Default write:
  Files : 4
  Size  : 4.6 MB
  Rows  : 200,000



26/04/10 21:05:48 WARN TaskSetManager: Stage 9 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


  mode=overwrite  → 100 rows


26/04/10 21:05:50 WARN TaskSetManager: Stage 16 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  mode=append     → 200 rows


26/04/10 21:05:51 WARN TaskSetManager: Stage 23 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  mode=ignore     → 200 rows
  mode=error      → AnalysisException (path exists)


26/04/10 21:05:52 WARN TaskSetManager: Stage 28 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


## Step 2 — Compression Codecs

| Codec | Ratio | Speed | CPU | Use when |
|---|---|---|---|---|
| `snappy` | Medium | Fast | Low | Default — balanced for hot data |
| `zstd` | High | Medium | Medium | Long-term storage, cold data |
| `gzip` | High | Slow | High | Archival, rarely queried |
| `lz4` | Low | Very fast | Very low | Temporary files, fast pipelines |
| `none` | 1x | Fastest | Zero | When CPU is the bottleneck |
| `brotli` | Very high | Slow | High | Maximum compression |


In [3]:
codecs = ["snappy", "zstd", "gzip", "lz4", "none"]
results = {}

for codec in codecs:
    path = f"{OUT}/codec_{codec}"
    t0 = time.time()
    df.write.mode("overwrite").option("compression", codec).parquet(path)
    t_write = time.time() - t0

    files = glib.glob(f"{path}/*.parquet")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024

    t0 = time.time()
    spark.read.parquet(path).agg(F.sum("revenue")).collect()
    t_read = time.time() - t0

    results[codec] = {"write": t_write, "size": size_mb, "read": t_read}

base_size = results["none"]["size"]
print(f"{'Codec':<10} {'Write':>8} {'Size MB':>10} {'vs none':>8} {'Read':>8}")
print("-" * 50)
for codec, r in results.items():
    ratio = f"{r['size']/base_size:.2f}x"
    print(f"  {codec:<8} {r['write']:>6.2f}s {r['size']:>8.1f} MB {ratio:>8} {r['read']:>6.3f}s")

print()
print("Recommendation: snappy for hot data, zstd for storage, lz4 for temp files")

26/04/10 21:05:52 WARN TaskSetManager: Stage 29 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:05:53 WARN TaskSetManager: Stage 34 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:05:55 WARN TaskSetManager: Stage 39 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:05:56 WARN TaskSetManager: Stage 44 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:05:58 WARN TaskSetManager: Stage 49 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Codec         Write    Size MB  vs none     Read
--------------------------------------------------
  snappy     1.17s      4.6 MB    0.64x  0.515s
  zstd       1.06s      3.1 MB    0.44x  0.429s
  gzip       1.13s      3.2 MB    0.45x  0.406s
  lz4        0.92s      4.6 MB    0.63x  0.373s
  none       0.92s      7.2 MB    1.00x  0.342s

Recommendation: snappy for hot data, zstd for storage, lz4 for temp files


## Step 3 — Row Group Size: Granularity vs Throughput

Row group size is the most important Parquet write parameter.
It controls the granularity of **data skipping**.

```
Small row groups (8 MB):
  + More precise skipping for selective queries
  − Larger footer, more metadata overhead
  − Less effective compression (fewer values to compress together)

Large row groups (256 MB):
  + Better compression ratio
  + Less overhead for full scans
  − Less precise skipping — must read more even for selective queries
```
Default is 128 MB — a good balance for most workloads.


In [4]:
row_group_sizes = [
    ("8mb",   8  * 1024 * 1024),
    ("64mb",  64 * 1024 * 1024),
    ("128mb", 128* 1024 * 1024),   # default
    ("256mb", 256* 1024 * 1024),
]

print(f"{'Config':<10} {'Size MB':>10} {'Files':>8} {'Full scan':>12} {'Filter scan':>14}")
print("-" * 60)

for label, rg_size in row_group_sizes:
    path = f"{OUT}/rg_{label}"
    df.write.mode("overwrite") \
            .option("parquet.block.size", str(rg_size)) \
            .option("compression", "snappy") \
            .parquet(path)

    files   = glib.glob(f"{path}/*.parquet")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024

    t0 = time.time()
    spark.read.parquet(path).agg(F.sum("revenue")).collect()
    t_full = time.time() - t0

    t0 = time.time()
    spark.read.parquet(path) \
              .filter(F.col("category") == "Electronics") \
              .agg(F.sum("revenue")).collect()
    t_filter = time.time() - t0

    print(f"  {label:<8} {size_mb:>8.1f} MB {len(files):>6} {t_full:>10.3f}s {t_filter:>12.3f}s")

print()
print("Tuning guide:")
print("  Analytical (full scans)   → 128-256 MB row groups")
print("  Point lookups / selective → 32-64 MB row groups")
print("  Streaming (small batches) → match batch size")

Config        Size MB    Files    Full scan    Filter scan
------------------------------------------------------------


26/04/10 21:05:59 WARN TaskSetManager: Stage 54 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  8mb           4.6 MB      4      0.339s        0.477s


26/04/10 21:06:01 WARN TaskSetManager: Stage 63 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


  64mb          4.6 MB      4      0.338s        0.383s


26/04/10 21:06:03 WARN TaskSetManager: Stage 72 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  128mb         4.6 MB      4      0.331s        0.359s


26/04/10 21:06:04 WARN TaskSetManager: Stage 81 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  256mb         4.6 MB      4      0.336s        0.342s

Tuning guide:
  Analytical (full scans)   → 128-256 MB row groups
  Point lookups / selective → 32-64 MB row groups
  Streaming (small batches) → match batch size


## Step 4 — Controlling Output Files: coalesce vs repartition


In [5]:
print(f"Default partitions: {df.rdd.getNumPartitions()}")
print()

for method, n_parts in [("coalesce(1)", 1), ("coalesce(4)", 4), ("repartition(8)", 8)]:
    if "coalesce" in method:
        writer = df.coalesce(int(method.split("(")[1].rstrip(")")))
    else:
        writer = df.repartition(int(method.split("(")[1].rstrip(")")))

    path = f"{OUT}/parts_{n_parts}"
    t0 = time.time()
    writer.write.mode("overwrite").parquet(path)
    t_write = time.time() - t0

    files   = glib.glob(f"{path}/*.parquet")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
    print(f"  {method:<20} → {len(files)} file(s), {size_mb:.1f} MB, write: {t_write:.2f}s")

print()
print("coalesce(1)  : single file, minimal shuffle — good for small exports")
print("coalesce(4)  : fewer files, partial shuffle — reduce small file problem")
print("repartition  : full shuffle, even distribution — use for large data")

Default partitions: 4



26/04/10 21:06:06 WARN TaskSetManager: Stage 90 contains a task of very large size (12425 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  coalesce(1)          → 1 file(s), 4.6 MB, write: 1.06s


26/04/10 21:06:07 WARN TaskSetManager: Stage 91 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  coalesce(4)          → 4 file(s), 4.6 MB, write: 0.92s


26/04/10 21:06:08 WARN TaskSetManager: Stage 92 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  repartition(8)       → 8 file(s), 4.7 MB, write: 1.63s

coalesce(1)  : single file, minimal shuffle — good for small exports
coalesce(4)  : fewer files, partial shuffle — reduce small file problem
repartition  : full shuffle, even distribution — use for large data


## Step 5 — Writing Sorted Data: Maximising Data Skipping

When data is sorted by a filter column before writing, each row group
contains a contiguous range of that column's values.
This maximises the effectiveness of min/max statistics for skipping.


In [6]:
# Unsorted write
unsorted_path = f"{OUT}/unsorted"
df.write.mode("overwrite").parquet(unsorted_path)

# Sorted by category (most common filter column)
sorted_path = f"{OUT}/sorted_by_category"
df.orderBy("category","country","order_date") \
  .write.mode("overwrite").parquet(sorted_path)

# Benchmark selective query
runs = 3
for label, path in [("Unsorted", unsorted_path), ("Sorted by category", sorted_path)]:
    times = []
    for _ in range(runs):
        t0 = time.time()
        spark.read.parquet(path) \
             .filter(F.col("category") == "Electronics") \
             .agg(F.sum("revenue"), F.count("*")).collect()
        times.append(time.time() - t0)
    median = sorted(times)[1]
    print(f"  {label:<25} {median:.3f}s")

print()
print("Sorting before write enables row group skipping:")
print("  Sorted   → each row group has 1 category → skip 5/6 row groups")
print("  Unsorted → each row group has all categories → skip nothing")
print()
print("Note: sorting adds write time but pays off on repeated reads")

26/04/10 21:06:09 WARN TaskSetManager: Stage 95 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:06:10 WARN TaskSetManager: Stage 96 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:06:10 WARN TaskSetManager: Stage 97 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  Unsorted                  0.359s
  Sorted by category        0.345s

Sorting before write enables row group skipping:
  Sorted   → each row group has 1 category → skip 5/6 row groups
  Unsorted → each row group has all categories → skip nothing

Note: sorting adds write time but pays off on repeated reads


## Step 6 — repartitionByRange: Smart Pre-Sort for Writing


In [7]:
# repartitionByRange sorts AND partitions data before write
# Better than orderBy + write because it's parallelized
range_path = f"{OUT}/range_partitioned"

t0 = time.time()
df.repartitionByRange(4, "category", "country") \
  .write.mode("overwrite").parquet(range_path)
t_range = time.time() - t0

files = glib.glob(f"{range_path}/*.parquet")
print(f"repartitionByRange(4, category, country):")
print(f"  Files     : {len(files)}")
print(f"  Write time: {t_range:.2f}s")
print()

# Each file contains a range of (category, country) values
try:
    import pyarrow.parquet as pq
    print("Row group stats per file (first 4):")
    for pf in sorted(files)[:4]:
        meta = pq.ParquetFile(pf).metadata
        rg   = meta.row_group(0)
        cat_col = rg.column(3)   # category
        if cat_col.statistics and cat_col.statistics.has_min_max:
            print(f"  {pf.split('/')[-1][-20:]}: "
                  f"category [{cat_col.statistics.min} → {cat_col.statistics.max}]  "
                  f"rows={rg.num_rows:,}")
except ImportError:
    print("pyarrow not available for metadata inspection")

26/04/10 21:06:14 WARN TaskSetManager: Stage 124 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:06:14 WARN TaskSetManager: Stage 125 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


repartitionByRange(4, category, country):
  Files     : 4
  Write time: 1.25s

Row group stats per file (first 4):
  -c000.snappy.parquet: category [Books → Clothing]  rows=58,346
  -c000.snappy.parquet: category [Clothing → Food]  rows=45,929
  -c000.snappy.parquet: category [Food → Furniture]  rows=45,819
  -c000.snappy.parquet: category [Furniture → Sports]  rows=49,906


## Summary: Parquet Write Best Practices

```python
# Production write — good defaults
df.write
  .mode("overwrite")
  .option("compression",       "zstd")        # best ratio for storage
  .option("parquet.block.size", 134217728)    # 128 MB row groups (default)
  .parquet("/path/output")

# For repeated selective queries — sort before write
df.orderBy("filter_column")
  .write.mode("overwrite").parquet("/path/")

# For large exports — repartitionByRange for even distribution
df.repartitionByRange(n_files, "partition_col")
  .write.mode("overwrite").parquet("/path/")

# For tools that expect single file
df.coalesce(1).write.mode("overwrite").parquet("/path/")
```

| Goal | Setting |
|---|---|
| Best compression | `compression=zstd` |
| Fastest write | `compression=lz4` or `none` |
| Best selective read | Sort before write + small row groups |
| Best full scan | Large row groups (256 MB) |
